In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np

In [ ]:
import pysem

In this notebook, we are creating SEM3D input files to run a simulation in a simple cubic domain with homogeneous properties.

In [ ]:
# folder to store the input files
path_folder = './tutorial1/'

# 1. Physical domain

The 3D domain is defined as a cube $[x_{min}, x_{max}] \times [y_{min}, y_{max}] \times [z_{min}, z_{max}]$. Each axis is discretized in elements of size $dx \times dy \times dz$. 

In [ ]:
xmin = -1000
xmax = 1000
dx = 100

ymin = -1000
ymax = 1000
dy = 100

zmin = -1000
zmax = 1000
dz = 100

# 2. Geological properties

We use a simple model with one single layer (`Nlayers=1`). Each layer is defined by:
- celerity of P waves (`Vp`)
- celerity of S waves (`Vs`)
- density (`Ds`)
- quality factor of P waves (`Qp`)
- quality factor of S waves (`Qs`)

In [ ]:
Nlayers = 1
Vp = np.array([6300])
Vs = np.array([2500])
Ds = np.array([2800])
Qp = np.array([1000])
Qs = np.array([1000])

# 3. Simulation parameters

We define the total simulation time (`sim_time`) and the coordinates of the source (`source_coords`).

In [ ]:
sim_time = 6 # seconds to simulate
source_coords = np.array([0, 0, 0]) # x, y, z
# moment tensor source components
moment = {'xx': 1.0, 'yy': 1.0, 'zz': 1.0,
          'xy': 0.0, 'xz': 0.0, 'yz': 0.0,}
amplitude = 1e18 # seismic moment magnitude

# 4. Write the input files

`mat.dat` describes the physical domain. To simulate an infinite domain, we put absorbing boundary conditions (Perfectly Matched Layers) on each boundary.

In [ ]:
lines = [f'{xmin}\t # xmin',
         f'{xmax}\t # xmax',
         f'{dx}\t # xstep',
         f'{ymin}\t # ymin',
         f'{ymax}\t # ymax',
         f'{dy}\t # ystep',
         f'{zmax}\t # zmax',
         f'{Nlayers}\t # nb. of layers',
         f'{zmax-zmin} {int(np.ceil((zmax-zmin)/dz))}\t # upper layer: thickness and nb of steps',
         '2           # nb of PMLs (0 = no PML)',
         '1 1         # PMLs on top? at the bottom? (0: no, 1: yes)',
         '1           # 8 or 27 control points for elements (1 or 2)']

with open(os.path.join(path_folder,'mat.dat'), 'w') as f:
    f.writelines([string + '\n' for string in lines])

`mater.in` describes the geological properties

In [ ]:
lines = [f'{Nlayers} # number of layers',
         '# Domain Vp[m/s] Vs[m/s] Ds[Kg/m3] Qp Qs']
for i in range(Nlayers):
    lines.append(f'S {Vp[i]} {Vs[i]} {Ds[i]} {Qp[i]} {Qs[i]}')
    
with open(os.path.join(path_folder,'mater.in'), 'w') as f:
    f.writelines([string + '\n' for string in lines])

Write the coordinates of stations in `stations.txt` to record ground motion. Sensors should not be located to close from the boundary to avoid recording unwanted reflections that are not prefectly absorbed by the PML. As a rule of thumb, keep a distance $dx$ from the boundary. 

Here, we create a regular grid of sensors just below the surface. 

In [ ]:
from pysem import create_stations

In [ ]:
lims = {'xmin':xmin+dx, 
        'xmax':xmax-dx,
        'nx':10, # number of sensors in the x direction
        'ymin':ymin+dy, 
        'ymax':ymax-dy,
        'ny':10,
        'zmin':zmin+dz, # sensors only at the surface
        'zmax':zmax-dz,
        'nz':10}

# create the grid of coordinates
grd = create_stations.grid(lims)

# write the file
create_stations.write_stations(grd, fn=os.path.join(path_folder,'stations.txt'))

MonitorSetName = "Stations"

`input.spec` contains all parameters needed to run the simulation. For most of the parameters, you don't need to change them. The main parameters you may want to modify are:
- the simulation time
- the source coordinates

In [ ]:
lines = ['# -*- mode: perl -*-',
         'run_name = "tutorial1";',
         '',
        '# duration of the run',
        f'sim_time = {sim_time};',
        'mesh_file = "mesh4spec"; # input mesh file',
        'mat_file = "material.input";',
        'dim=3;',
        'ngll=5;',
        '',
        'snapshots {',
        '    save_snap = true;',
        '    snap_interval = 0.1;',
        '    select material = 0;',
        '};',
        '',
        '# Description des capteurs',
        'save_traces = true;',
        'traces_format = hdf5;',
        f'capteurs "{MonitorSetName}"'+' {',
        '    type = points;',
        '    file = "stations.txt";',
        '    period = 1;',
        '};',
        '',
        '# Fichier protection reprise',
        'prorep=false;',
        'prorep_iter=1000;',
        '',
        '# introduce a source',
        'source {',
        '    # coordinates of the sources ((x,y,z) or (lat,long,R) if rotundity is considered)',
        f'    coords = {source_coords[0]} {source_coords[1]} {source_coords[2]};',
        '    # Type (impulse, moment tensor, fluidpulse)',
        '    type = moment;',
        '    # Direction 0.x,1.y ou 2.z (only for Impulse)',
        f'    moment = {moment['xx']} {moment['yy']} {moment['zz']} {moment['xy']} {moment['xz']} {moment['yz']};',
        '    # Function 1.gaussian,2.ricker,3.tf_heaviside,4.gabor,5.file,6.spice_bench,7.sinus, or external file',
        '    func = gaussian;',
        '    ts = 0.1;',
        '    tau = 0.03;',
        f'    amplitude = {amplitude};',
        '};',
        '',
        'time_scheme {',
        '    accel_scheme = false;  # Acceleration scheme for Newmark',
        '    veloc_scheme = true;   # Velocity scheme for Newmark',
        '    alpha = 0.5;           # alpha (Newmark parameter)',
        '    beta = -0.5;           # beta (Newmark parameter)',
        '    gamma = 1;             # gamma (Newmark parameter)',
        '    courant=0.2;',
        '};',
        '',
        'amortissement {',
        '    nsolids = 0;           # number of solids for attenuation (0 if no attenuation)',
        '    atn_band = 10  0.05;   # attenuation period band',
        '    atn_period = 0.2;      # model period ',
        '};']

with open(os.path.join(path_folder,'input.spec'), 'w') as f:
    f.writelines([string + '\n' for string in lines])

# 5. Launch the simulation

You need to copy and paste the file `mesh.input` that defines the number of processors used to partition the mesh. This number must be the same as the number of processors used to run the simulation (4 in this tutorial). 

Send all the created files to the cluster with the `scp` command: `scp my_laptop/path/to/folder/tutorial1 mylogin@chome.metz.supelec.fr:/path/to/my/sem3d/tutorial1`

Run `MESHER.sbatch` to create the mesh. 

Then run `SOLVER.sbatch`

# 6. Post-process the results

Once the simulation is done, you should have one folder `traces` containing the timeseries of displacement, velocity, and acceleration at each sensor. In the `res` folder, you should have the snapshots of all properties for each grid point. 

- Using `Paraview`, you can visualize the snapshots

- By running `parse_sem3d_traces.py`, you can plot the time series by sensor set name `MonitorSetName`.

In [ ]:
! python3 ./pysem/parse_sem3d_traces.py @@wkdir {os.path.join(path_folder,'traces')} @@format h5 @@names {MonitorSetName} @@variables Veloc @@components x y z @@monitors -1 @@p

In [15]:
from pysem.parse_sem3d_traces import ParseSEM3DH5Traces

# Define your options as a dictionary
options = {
    'wkdir': os.path.join(path_folder,'traces'),
    'format': 'h5',
    'names': [MonitorSetName],
    'variables': ['Veloc'],
    'components': ['x', 'y', 'z'],
    'monitors':-1
}

# 1. Parse the data
# This returns a dictionary of SEM3DMonitor objects
stream = ParseSEM3DH5Traces(**options)

# 2. Access a specific monitor set
my_monitor = stream['Uobs']


# 3. Interactive Plotting (shows directly in the notebook)
# Ensure %matplotlib inline is set if you want to see it here
import matplotlib.pyplot as plt
%matplotlib inline

# Plot station 0
fig = my_monitor.Plot(wkdir='./', variables=['Veloc'], components=['x'], monitors=[0])
plt.show()

IndexError: list index out of range